In [1]:
%load_ext autoreload
%autoreload 2

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

import main_analysis as ma

# Plot settings
import matplotlib as mpl
from matplotlib import rcParams
rcParams.update({'figure.autolayout': False})
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'Avenir'

RESULTS_DIR = 'results'
MODELS_DIR = 'models'
PLOTS_DIR = f'{RESULTS_DIR}/plots/preprint'
os.makedirs(PLOTS_DIR, exist_ok=True)

pixdim[1,2,3] should be non-zero; setting 0 dims to 1


# Main figure (TODO)

# Supplementary figures

## Fig S9: Lesioning across across ANN t-thresholds

In [9]:
savefig = True

plot_dir = f'{PLOTS_DIR}/section5/supplementary_figures'
os.makedirs(plot_dir, exist_ok=True)

SUBJECTSUBJECT_CORRELATIONS_CSV_PATH = f'{RESULTS_DIR}/subjectsubject_correlations_nsd515.csv'
lesion_csv_path = f'{RESULTS_DIR}/nsd515_lesion_ems_fixedalpha.csv'
lesion_csv = pd.read_csv(lesion_csv_path)

fmri_tthreshold = 7

results_df = lesion_csv.copy()
subjectsubject_correlations_csv = pd.read_csv(SUBJECTSUBJECT_CORRELATIONS_CSV_PATH)
subjects = results_df['subject'].unique().tolist()

frois = ['FFA', 'PPA', 'EBA']
ordered_tthresholds = [-1, 11, 9, 7, 5, 3]
xticks_dict = {
    -1: 'Intact',
    11: '11',
    9: '9',
    7: '7',
    5: '5',
    3: '3'
}
metrics = ['normed_pearsonr', 'normed_rdmr']
plot_ceiling = False

x_offset = 0.05
alpha = 0.5
markersize = 25
x = [-0.2, 0.3, 0.6, 0.9, 1.2, 1.5]
fig, axs = plt.subplots(len(metrics), len(frois), figsize=(5*len(frois), 5*len(metrics)), dpi=300)

colors = [
    '#ffffff',  # white for intact
    '#ffcccc',  # very light red
    '#ff9999',  # light red
    '#ff0000',  # pure red (middle)
    '#e60000',  # slightly darker red
    '#cc0000',  # darker red
]

for midx, metric in enumerate(metrics):
    for fidx, froi in enumerate(frois):
        print(metric, froi)
        # For specific froi & fmri_tthreshold, get results_df
        results_df_froi = results_df[
            (results_df['model_froi'] == froi) &
            (results_df['fmri_froi'] == froi) &
            (results_df['fmri_tthreshold'] == fmri_tthreshold)
        ]
        # Take median across subjects for each model, model_threshold
        results_df_froi_sbjmedian = results_df_froi.groupby(
            ['model', 'model_tthreshold']
        ).agg({
            'raw_pearsonr': 'median',
            'normed_pearsonr': 'median',
            'raw_rdmr': 'median',
            'normed_rdmr': 'median',
            'n_units_layer': 'mean',
            'n_units_lesioned': 'mean',
            'percent_units_lesioned': 'mean',
        }).reset_index()

        if plot_ceiling:
            ceiling_values = []
            for subject1_idx in range(len(subjects)):
                for subject2_idx in range(subject1_idx, len(subjects)):
                    subject1 = subjects[subject1_idx]
                    subject2 = subjects[subject2_idx]
                    if subject1 == subject2:
                        continue
                    subjectsubject_froi = subjectsubject_correlations_csv[
                        (subjectsubject_correlations_csv['fmri_froi'] == froi) &
                        (subjectsubject_correlations_csv['subject1'] == subject1) &
                        (subjectsubject_correlations_csv['subject2'] == subject2) & 
                        (subjectsubject_correlations_csv['fmri_tthreshold'] == 7)
                    ]
                    ceiling_metric_ = 'pearsonr' if metric == 'raw_pearsonr' else metric
                    ceiling_values += list(subjectsubject_froi[ceiling_metric_])
                    # print(f'Subject {subject1} vs Subject {subject2}: {subjectsubject_froi[ceiling_metric]}')
            ceiling_values = np.array(ceiling_values).flatten()
            # print(f'Ceiling values: {ceiling_values}')
            ceiling_median = np.median(ceiling_values)
            ceiling_sem = np.std(ceiling_values) / np.sqrt(len(ceiling_values))

            # Plot horizontal line at median of ceiling values
            plt.axhline(y=ceiling_median, color='royalblue', linestyle='-', alpha=0.8, linewidth=1, zorder=1)

            # Fill area around the line for SEM
            plt.fill_between(
                [x[0]-0.5, x[1]+0.5],  # Extend across full x range
                ceiling_median - ceiling_sem,
                ceiling_median + ceiling_sem,
                color='royalblue', alpha=0.2, zorder=0, edgecolor='none'
            )

        
        # model_tthresholds = results_df_froi_sbjmedian['model_tthreshold'].unique()
        model_tthresholds = ordered_tthresholds
        assert set(model_tthresholds) == set(ordered_tthresholds), \
            f"Model tthresholds {model_tthresholds} are same as ordered tthresholds {ordered_tthresholds}"
        n_model_tthresholds = len(model_tthresholds)

        # percent_lesioned_means = []
        # percent_lesioned_sems = []

        xticklabels = []

        for tidx, model_tthreshold in enumerate(model_tthresholds):
            tthreshold_results = results_df_froi_sbjmedian[
                (results_df_froi_sbjmedian['model_tthreshold'] == model_tthreshold)
            ]
            y_pts = tthreshold_results[metric].values
            percent_lesioneds = tthreshold_results['percent_units_lesioned'].values
            pl_mean = np.mean(percent_lesioneds)
            pl_sem = np.std(percent_lesioneds) / np.sqrt(len(percent_lesioneds))
            # percent_lesioned_means.append(np.mean(percent_lesioneds))
            # percent_lesioned_sems.append(np.std(percent_lesioneds) / np.sqrt(len(percent_lesioneds)))
            # Add text on top of each x point mentioning the percent lesioned mean +- sem
            # axs[midx, fidx].text(
            #     x[tidx] + x_offset, 0.95,
            #     f"{pl_mean:.1f} ± {pl_sem:.1f}%",
            #     horizontalalignment='center', verticalalignment='bottom',
            #     fontsize=8, color='black', alpha=0.8
            # )
            # instead of text, make it x tick labels
            if model_tthreshold == -1:
                xticklabels.append('0%\n(All units)')
            else:
                xticklabels.append(f"{pl_mean:.1f}%\n(t > {model_tthreshold})")
        
            x_froi = (np.ones(len(y_pts)) * x[tidx]) - x_offset
            axs[midx, fidx].scatter(
                x_froi, y_pts, color='gray', alpha=alpha, zorder=2,
                edgecolors='k', s=markersize
            )
            # connect points with a line
            # for i in range(len(froi_pts)):
            #     axs[midx, fidx].plot([x_froi[i], x_lesion[i]], [froi_pts[i], lesion_pts[i]], color='red', alpha=0.2, linewidth=0.5, zorder=1)

            # Bar plot with median and SEM
            bar_width = 0.2 #0.4
            median = np.median(y_pts)
            sem = y_pts.std() / np.sqrt(len(y_pts))

            axs[midx, fidx].bar(
                x[tidx], median, yerr=sem, width=bar_width, color=colors[tidx],
                alpha=1, capsize=0, error_kw={'linewidth': 1}, edgecolor='k'
            )
            # # Conduct Welch's t-test
            # t_value, p_value = scipy.stats.ttest_ind(froi_pts, lesion_pts, equal_var=False)
            # # axs[midx, fidx].text(0.95, 0.89,
            # #          f"t-value: {t_value:.2f}\np-value: {p_value:.2e}",
            # #          horizontalalignment='right', verticalalignment='bottom',
            # #          transform=axs[midx, fidx].transAxes)
            # print(f"t-value: {t_value:.2f}\np-value: {p_value:.2e}")
        axs[midx, fidx].text(0.95, 0.89,
                    froi,
                    horizontalalignment='right', verticalalignment='bottom',
                    transform=axs[midx, fidx].transAxes)

        axs[midx, fidx].axhline(y=0, color='black', linestyle='--', alpha=1, linewidth=1)

        axs[midx, fidx].spines['top'].set_visible(False)
        axs[midx, fidx].spines['right'].set_visible(False)
        axs[midx, fidx].spines['left'].set_position(('outward', 5))
        axs[midx, fidx].spines['bottom'].set_position(('outward', 5))
        axs[midx, fidx].set_yticks([0, 0.25, 0.5, 0.75, 1])
        axs[midx, fidx].spines['left'].set_bounds(0, 1)
        axs[midx, fidx].set_ylim(0, 1)
        # axs[midx, fidx].tick_params(axis='y', labelleft=False)
        axs[midx, fidx].spines['bottom'].set_bounds(x[0], x[-1])
        # xticklabels = [xticks_dict[t] for t in model_tthresholds]
        axs[midx, fidx].set_xticks(x, xticklabels, fontsize=10)
        # axs[midx, fidx].xticks(x, model_tthresholds)
        axs[midx, fidx].set_xlim(x[0]-0.25, x[-1]+0.25)
        # axs[midx, fidx].tick_params(axis='x', labelbottom=False)
        axs[midx, fidx].set_xlabel('Percentage of units lesioned', fontsize=12)
        if metric == 'normed_pearsonr':
            axs[midx, fidx].set_ylabel(f'univariate', fontsize=12)
        elif metric == 'normed_rdmr':
            axs[midx, fidx].set_ylabel(f'multivariate', fontsize=12)

        if froi != 'FFA':
            axs[midx, fidx].spines['left'].set_visible(False)
            # Stop showing y-axis ticks
            axs[midx, fidx].yaxis.set_ticks([])
            axs[midx, fidx].set_ylabel('')
        # plt.title(f'{froi} fROI, mapper: Ridge', fontsize=14, pad=10)

fig.supylabel(r"Prediction accuracy, $R$", fontsize=14)
plt.tight_layout()
if savefig:
    plt.savefig(f'{plot_dir}/figS9_EMlesions_acrosstthresholds.pdf', dpi=300, bbox_inches='tight', transparent=True)
else:
    plt.show()
plt.close()

normed_pearsonr FFA
normed_pearsonr PPA
normed_pearsonr EBA
normed_rdmr FFA
normed_rdmr PPA
normed_rdmr EBA


## Fig S10: L1-positive mapping